# Phase 1 — Outcome Prediction Model Demo

**Empirical (Option A) + XGBoost ML + Comparison**

This notebook demonstrates the full Phase 1 pipeline:

| Section | What it does |
|---------|-------------|
| 1 | Build & inspect matchup stats table |
| 2 | Empirical predictions for sample pairs |
| 3 | Train XGBoost + feature importances |
| 4 | XGBoost predictions for same pairs |
| 5 | Side-by-side comparison table (Empirical vs XGBoost vs Actual) |
| 6 | Leaderboard — best bowlers vs a batter |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'phase1'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
print('Imports OK')

---
## Section 1 — Matchup Stats Table

In [ ]:
from phase1.matchup_builder import build_matchup_table, save, OUTPUT_PATH

matchup_df = build_matchup_table()

print(f'\nTotal batter-bowler pairs : {len(matchup_df):,}')
print(f'Pairs with >= 10 balls    : {(matchup_df["balls"] >= 10).sum():,}')
print(f'Pairs with >= 20 balls    : {(matchup_df["balls"] >= 20).sum():,}')
print()

# Save (skip if already saved)
save(matchup_df)

matchup_df.nlargest(10, 'balls')[[
    'batter','bowler','balls','expected_runs','strike_rate','prob_W'
]]

In [ ]:
# Distribution of balls per pair
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(matchup_df['balls'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Balls in matchup')
axes[0].set_ylabel('Number of pairs')
axes[0].set_title('Distribution of Matchup Sizes')
axes[0].axvline(10, color='red', linestyle='--', label='>= 10 threshold')
axes[0].legend()

axes[1].hist(matchup_df[matchup_df['balls'] >= 5]['expected_runs'],
             bins=40, color='orange', edgecolor='white')
axes[1].set_xlabel('Expected runs per ball')
axes[1].set_ylabel('Number of pairs')
axes[1].set_title('Expected Runs Distribution (>= 5 balls)')

plt.tight_layout()
plt.show()

---
## Section 2 — Empirical Predictions

In [ ]:
from phase1.empirical_predictor import EmpiricalPredictor

emp = EmpiricalPredictor().load()
print('EmpiricalPredictor loaded.')

In [ ]:
SAMPLE_PAIRS = [
    ('MN Samuels', 'CJ Jordan'),    # high-ball matchup
    ('Q de Kock',  'MM Ali'),       # high-ball matchup
    ('V Kohli',    'JJ Bumrah'),    # sparse matchup
]

for batter, bowler in SAMPLE_PAIRS:
    r = emp.predict(batter, bowler)
    print(f"\n{batter} vs {bowler}")
    print(f"  Source  : {r.get('source','?')}  |  Balls: {r.get('balls_sample','?')}")
    print(f"  Probs   : { {k: f\"{v*100:.1f}%\" for k, v in r['probs'].items()} }")
    print(f"  Exp runs: {r['expected_runs']}  |  SR: {r['strike_rate']}  |  {r['tendency']}")

---
## Section 3 — Train XGBoost + Feature Importances

In [ ]:
from phase1.ml_predictor import MLPredictor

ml = MLPredictor()
train_result = ml.train()
print(f"\nXGBoost training complete!")
print(f"Overall accuracy: {train_result['accuracy']:.4f}")

In [ ]:
# Feature importance chart
imp = train_result['feature_importances']
labels = list(imp.keys())
values = [imp[k] for k in labels]
colors = ['#2196F3' if v == max(values) else '#90CAF9' for v in values]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(labels, values, color=colors, edgecolor='white')
ax.set_xlabel('Feature Importance (gain)')
ax.set_title('XGBoost Feature Importances')
ax.invert_yaxis()
for bar, val in zip(bars, values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## Section 4 — XGBoost Predictions

In [ ]:
for batter, bowler in SAMPLE_PAIRS:
    r = ml.predict(batter, bowler)
    print(f"\n{batter} vs {bowler}")
    print(f"  Method  : {r['method']}  |  Balls: {r.get('balls_sample','?')}")
    print(f"  Probs   : { {k: f\"{v*100:.1f}%\" for k, v in r['probs'].items()} }")
    print(f"  Exp runs: {r['expected_runs']}  |  SR: {r['strike_rate']}  |  {r['tendency']}")

---
## Section 5 — Comparison Table (Empirical vs XGBoost vs Actual)

In [ ]:
from phase1.predictor import Phase1Predictor

predictor = Phase1Predictor()

OUTCOMES = ['0', '1', '2', '3', '4', '6', 'W']

def actual_probs(batter, bowler, matchup_df):
    """Pull actual observed frequencies from matchup_stats."""
    mask = (matchup_df['batter'] == batter) & (matchup_df['bowler'] == bowler)
    row  = matchup_df[mask]
    if row.empty:
        return {o: None for o in OUTCOMES}
    r = row.iloc[0]
    return {
        '0': round(float(r['prob_0']) * 100, 1),
        '1': round(float(r['prob_1']) * 100, 1),
        '2': round(float(r['prob_2']) * 100, 1),
        '3': round(float(r['prob_3']) * 100, 1),
        '4': round(float(r['prob_4']) * 100, 1),
        '6': round(float(r['prob_6']) * 100, 1),
        'W': round(float(r['prob_W']) * 100, 1),
    }

rows = []
for batter, bowler in SAMPLE_PAIRS:
    cmp    = predictor.compare(batter, bowler)
    actual = actual_probs(batter, bowler, matchup_df)
    for item in cmp['outcomes']:
        o = item['outcome']
        rows.append({
            'Batter': batter, 'Bowler': bowler, 'Outcome': o,
            'Empirical %': item['empirical_pct'],
            'XGBoost %':   item['xgboost_pct'],
            'Actual %':    actual.get(o),
            'Balls':       cmp['balls_sample'],
        })

cmp_df = pd.DataFrame(rows)
cmp_df

In [ ]:
# Visual comparison for one pair
PAIR = ('MN Samuels', 'CJ Jordan')
sub  = cmp_df[(cmp_df['Batter'] == PAIR[0]) & (cmp_df['Bowler'] == PAIR[1])].copy()
sub  = sub.set_index('Outcome')

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(sub))
w = 0.25

bars1 = ax.bar([i - w for i in x], sub['Empirical %'], width=w,
               label='Empirical', color='#1976D2', alpha=0.85)
bars2 = ax.bar([i     for i in x], sub['XGBoost %'],  width=w,
               label='XGBoost',   color='#F57C00', alpha=0.85)
bars3 = ax.bar([i + w for i in x], sub['Actual %'],   width=w,
               label='Actual',    color='#388E3C', alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels(sub.index)
ax.set_xlabel('Outcome')
ax.set_ylabel('Probability (%)')
ax.set_title(f'{PAIR[0]} vs {PAIR[1]} — Empirical vs XGBoost vs Actual')
ax.legend()
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()

In [ ]:
# Summary comparison: Expected runs
summary_rows = []
for batter, bowler in SAMPLE_PAIRS:
    cmp = predictor.compare(batter, bowler)
    mask = (matchup_df['batter'] == batter) & (matchup_df['bowler'] == bowler)
    actual_er = None
    if not matchup_df[mask].empty:
        actual_er = round(float(matchup_df[mask].iloc[0]['expected_runs']), 3)
    summary_rows.append({
        'Batter': batter,
        'Bowler': bowler,
        'Balls': cmp['balls_sample'],
        'Emp Source': cmp['sources']['empirical'],
        'Emp ExpRuns': cmp['expected_runs']['empirical'],
        'XGB ExpRuns': cmp['expected_runs']['xgboost'],
        'Actual ExpRuns': actual_er,
        'Emp Tendency': cmp['tendency']['empirical'],
        'XGB Tendency': cmp['tendency']['xgboost'],
    })

pd.DataFrame(summary_rows)

---
## Section 6 — Leaderboard

In [ ]:
# Top 10 bowlers vs a batter (by wicket probability)
BATTER = 'Q de Kock'
print(f'Top 10 bowlers vs {BATTER} — ranked by wicket probability:\n')
predictor.leaderboard_bowlers(BATTER, n=10)

In [ ]:
# Top 10 batters vs a bowler (by expected runs)
BOWLER = 'MM Ali'
print(f'Top 10 batters vs {BOWLER} — ranked by expected runs per ball:\n')
predictor.leaderboard_batters(BOWLER, n=10)

In [ ]:
# Bar chart: wicket probability per bowler vs a batter
lb = predictor.leaderboard_bowlers(BATTER, n=10)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#D32F2F' if p > 0 else '#90A4AE' for p in lb['prob_W']]
ax.barh(lb['bowler'], lb['prob_W'] * 100, color=colors, edgecolor='white')
ax.set_xlabel('Wicket Probability (%)')
ax.set_title(f'Wicket Probability — Bowlers vs {BATTER}')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()

---
## Quick CLI usage

```bash
# From the models/phase1/ directory:
python predict_cli.py "MN Samuels" "CJ Jordan"
python predict_cli.py "V Kohli" --method empirical
python predict_cli.py "Q de Kock" --leaderboard
python predict_cli.py --bowler "MM Ali" --leaderboard
```